In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pygwalker as pyg
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os


In [2]:
#Importing feature_extraction.csv file

feat_df = pd.read_csv("../data/proccessed/feature_extraction_V3.csv")
#feat_df["max_complexity"].max()
feat_df.head()

,file name,loc,sloc,comment_ratio,blank_ratio,cyclomatic_complexity,max_complexity,maintainability_index,halstead_effort,num_functions,...,is_tested,dependency_count,commit_count,author_count,days_since_change,churn_rate,bug_fix_commits,refactor_commits,repo,label
0,wicked-garden/scripts\reset.py,222,169,0.085586,0.135135,9.666667,35,46.077070,1517.398123,6,...,0,6,1.0,1.0,1.122951,0.931646,1,1,wicked-garden,0
1,wicked-garden/scripts\resolve_path.py,22,8,0.363636,0.272727,0.000000,0,98.817997,2.377444,0,...,0,3,1.0,1.0,1.131148,0.060759,2,0,wicked-garden,1
2,wicked-garden/scripts\_agents.py,357,179,0.285714,0.198880,4.733333,20,50.826472,7470.287739,15,...,0,5,1.5,1.0,1.057377,1.521519,2,2,wicked-garden,1
3,wicked-garden/scripts\_capability_registry.py,226,196,0.106195,0.061947,1.000000,1,100.000000,0.000000,2,...,0,2,0.5,1.0,1.057377,0.572152,1,1,wicked-garden,0
4,wicked-garden/scripts\_capability_resolver.py,322,196,0.217391,0.180124,14.400000,27,49.128629,4896.134033,5,...,0,9,0.5,1.0,1.057377,0.815190,1,1,wicked-garden,0


I have anormal values here, since i will be using random forest (explain details later); these will interfere with the model training (make better notes for the complete version of this notebook)

In [3]:
type(feat_df["label"][0])

numpy.int64

In [4]:
feat_df.shape

(2503, 22)

In [5]:
print(feat_df["repo"].nunique())
print(feat_df.groupby("repo").size().sort_values())

22
repo
flow                             2
Discord-OTP-Forcer              11
ibet-Network                    12
hass-dab-pumps                  16
cli                             25
kernelci-pipeline               29
ibet-SmartContract              31
index-digest                    35
FreeCAD-DFM-Workbench           54
mesa-llm                        61
SpiceyPy                        63
draftjs_exporter                68
pyticktick                      68
simonwillisonblog              131
moldflow-api                   151
XMRT-Ecosystem                 153
FAIR                           161
penny                          191
wicked-garden                  201
pypto                          289
commander-spellbook-backend    292
TileOPs                        459
dtype: int64


In [6]:
cols_relativas = ["commit_count", "author_count", "churn_rate", "days_since_change"]
print(feat_df[cols_relativas].describe())

       commit_count  author_count   churn_rate  days_since_change
count   2503.000000   2503.000000  2503.000000        2503.000000
mean       2.305496      1.234579     2.789575           1.308596
std        4.442194      0.748839    12.133475           1.713341
min        0.111111      0.250000     0.000000           0.059372
25%        0.500000      1.000000     0.442053           0.962963
50%        1.000000      1.000000     1.000000           1.000000
75%        2.000000      1.000000     2.237742           1.156522
max       77.000000      8.000000   458.924370          27.892562


In [7]:
#Feat separation from label 
X = feat_df.iloc[:, 1:20]
Y = feat_df["label"]

Data manipulation

In [7]:

#limit_ch = feat_df["churn_rate"].quantile(0.99)
#feat_df["churn_rate"] = feat_df["churn_rate"].clip(upper=limit_ch)

limit_mc = feat_df["max_complexity"].quantile(0.99)
feat_df["max_complexity"] = feat_df["max_complexity"].clip(upper=limit_mc)
print(feat_df["max_complexity"].max())

42.98000000000002


In [15]:
limit_churn_rel = feat_df["churn_rate"].quantile(0.99)
feat_df["churn_rate"] = feat_df["churn_rate"].clip(upper=limit_churn_rel)
print("Nuevo máximo de churn_rate:", feat_df["churn_rate"].max())

limit_commit_rel = feat_df["commit_count"].quantile(0.99)
feat_df["commit_count"] = feat_df["commit_count"].clip(upper=limit_commit_rel)
print("Nuevo máximo de commit_count:", feat_df["commit_count"].max())

Nuevo máximo de churn_rate: 27.032633043478285
Nuevo máximo de commit_count: 20.934000000000058


In [16]:
print(feat_df[["commit_count", "author_count", "churn_rate", "days_since_change"]].describe())

       commit_count  author_count   churn_rate  days_since_change
count   2503.000000   2503.000000  2503.000000        2503.000000
mean       2.167642      1.234579     2.307478           1.308596
std        3.255630      0.748839     4.004520           1.713341
min        0.111111      0.250000     0.000000           0.059372
25%        0.500000      1.000000     0.442053           0.962963
50%        1.000000      1.000000     1.000000           1.000000
75%        2.000000      1.000000     2.237742           1.156522
max       20.934000      8.000000    27.032633          27.892562


In [17]:
excluir = ["file name", "repo", "bug_fix_commits", "refactor_commits", "label"]
features_v2 = [c for c in feat_df.columns if c not in excluir]

In [18]:
X = feat_df[features_v2]
Y = feat_df["label"] 

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [20]:
modelo_v2_relativo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_v2_relativo.fit(X_train, y_train)
# Evaluación
y_pred = modelo_v2_relativo.predict(X_test)

print("\nScore:", modelo_v2_relativo.score(X_test, y_test))
print()
print(classification_report(y_test, y_pred))
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print()

importancias = pd.Series(modelo_v2_relativo.feature_importances_, index=features_v2)
print(importancias.sort_values(ascending=False))


Score: 0.874251497005988

              precision    recall  f1-score   support

           0       0.86      0.95      0.90       311
           1       0.89      0.76      0.82       190

    accuracy                           0.87       501
   macro avg       0.88      0.85      0.86       501
weighted avg       0.88      0.87      0.87       501

Matriz de confusión:
[[294  17]
 [ 46 144]]

commit_count             0.292777
author_count             0.086052
churn_rate               0.085713
days_since_change        0.070146
dependency_count         0.066815
loc                      0.052173
sloc                     0.050192
blank_ratio              0.042401
avg_function_len         0.039619
maintainability_index    0.035670
halstead_effort          0.034310
cyclomatic_complexity    0.031979
comment_ratio            0.030788
num_functions            0.030113
max_complexity           0.022540
max_nesting_depth        0.019931
is_tested                0.008780
dtype: float64


In [21]:
os.makedirs("../models", exist_ok=True)

joblib.dump(modelo_v2_relativo, "../models/random_forest_v2.pkl")
joblib.dump(features_v2, "../models/features_v2.pkl")

print("Guardado: random_forest_v2.pkl y features_v2.pkl")

Guardado: random_forest_v2.pkl y features_v2.pkl
